# Comparing a general and a Singapore-adapted speech recogniser on Singapore English

**Research question:** Does Singapore-specific adaptation improve ASR (automatic speech recognition) of Singapore English, particularly Singapore English discourse particles (*lah, leh, lor, meh, ah*)?

**System A:** `openai/whisper-small` **System B:** `ivabojic/whisper-small-sing2eng-transcribe`

This notebook is my complete, reproducible pipeline. Both models are downloaded and run on the Colab machine itself (no closed APIs, no fine-tuning, no changes to the models). It produces numbers, tables and charts only; my interpretation is in my report.

**To reproduce:** (1) choose *Runtime → Change runtime type → T4 GPU*; (2) upload `data.zip` (containing `metadata.csv` and a folder `test_set` with the audio files) to the Files panel on the left; (3) fill in the licence details in Section 0; (4) choose *Runtime → Run all*.

## 0. Data source and licence
The data is released under the Singapore Open Data Licence v1.0, which requires a visible notice saying where it came from. Here I fill in the dataset name, the date I accessed it and its source. The notice is printed here, saved with my results and printed under each chart. The full licence is in the Appendix at the end.

In [ ]:
DATASET_NAME   = "IMDA - National Speech Corpus"   
DATE_ACCESSED  = "25 September 2026"  
DATASET_SOURCE = "nsc@imda.gov.sg"   
LICENCE_URL    = "https://data.gov.sg/open-data-licence"

# The notice in the format the licence asks for
LICENCE_NOTICE = (f"[Contains information from {DATASET_NAME} accessed on {DATE_ACCESSED} from {DATASET_SOURCE} "
                  f"which is made available under the terms of the Singapore Open Data Licence version 1.0 {LICENCE_URL}]")
print(LICENCE_NOTICE)
if "<<" in LICENCE_NOTICE:
    print("REMINDER: fill in the three dataset details above.")

## 1. Installing, importing and settings
I install the two packages Colab does not already have: `jiwer` (counts word errors) and `num2words` (turns digits like "3" into "three" when cleaning text). Then I import the tools I use, and put all my settings in one place: file locations, the two model names, my two conditions, the five target particles, and my annotation marks. I also record whether a GPU is available.

In [ ]:
!pip install -q jiwer num2words

import os, re, zipfile
import pandas as pd                    # tables
import matplotlib.pyplot as plt        # charts
import librosa                         # reads audio files
import torch                           # runs the models
import jiwer                           # counts word errors
from num2words import num2words        # 3 -> "three"
from transformers import WhisperProcessor, WhisperForConditionalGeneration   # the Whisper models

# ---------- where my files are ----------
DATA_ZIP      = "/content/data.zip"                 # the zip I uploaded to the Files panel
DATA_DIR      = "/content/data"                     # it is unpacked here
METADATA_FILE = f"{DATA_DIR}/metadata.csv"          # columns: file, speaker, condition, reference
AUDIO_DIR     = f"{DATA_DIR}/test_set"              # the audio files
OUTPUT_DIR    = "/content/outputs"                  # all results are saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- what I am testing ----------
SYSTEMS = {"A": "openai/whisper-small", "B": "ivabojic/whisper-small-sing2eng-transcribe"}
LABEL   = {"A": "System A (openai/whisper-small)", "B": "System B (ivabojic/whisper-small-sing2eng-transcribe)"}
CONDITIONS = ["particle_present", "particle_absent"]
PARTICLES  = ["oh", "lah", "leh", "lor", "meh", "ah"]

# Settings used by BOTH models when they transcribe (same settings keeps the comparison fair)
GEN_SETTINGS = dict(language="en",       # the speech is English
                    task="transcribe",   # write down what was said; do NOT translate
                    num_beams=1,         # always take the single most likely next word
                    do_sample=False,     # no randomness, so a re-run gives the same answer
                    max_new_tokens=225)  # upper limit on transcript length

# ---------- my annotation marks in the reference transcripts ----------
# Each pattern finds one kind of mark and captures the words inside it.
# I KEEP the words and remove only the marks.
ANNOTATIONS = {
    "name (#...#)":               re.compile(r"#([^#]*)#"),
    "code-switched (<...>)":      re.compile(r"<([^<>]*)>"),
    "discourse particle ([...])": re.compile(r"\[([^\[\]]*)\]"),
    "filler ((...))":             re.compile(r"\(([^()]*)\)"),
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU, so transcription will be slow. Use Runtime -> Change runtime type -> T4 GPU.")

## 2. Loading the test set and checking it
I unpack `data.zip`, read `metadata.csv` (one row per utterance: file, speaker, condition, reference) and open every audio file from the `test_set` folder, converting it to 16,000 measurements per second (what Whisper expects). Before any transcription I check: that no information is missing, that no file ID appears twice, that the condition labels are only my two conditions, that every annotation mark in a reference that is opened is also closed, and that every audio file exists and can be opened. Then I print the number of utterances, the number of speakers, the utterances per condition and any problems. Nothing is dropped silently: all problems are listed and saved to `data_check_problems.csv`.

In [ ]:
# ---------- unpack the zip ----------
if not os.path.exists(DATA_ZIP):
    raise FileNotFoundError("data.zip was not found. Upload it to the Files panel (left side) and run again.")
with zipfile.ZipFile(DATA_ZIP) as z:
    z.extractall(DATA_DIR)
if not os.path.isfile(METADATA_FILE) or not os.path.isdir(AUDIO_DIR):
    raise FileNotFoundError(f"Expected metadata.csv and a test_set folder inside data.zip, but found: {os.listdir(DATA_DIR)}")

# ---------- read the metadata ----------
meta = pd.read_csv(METADATA_FILE, dtype=str)           # read everything as text
meta = meta[["file", "speaker", "condition", "reference"]]
meta = meta.apply(lambda col: col.str.strip())         # remove stray spaces at the start/end

# ---------- check the metadata ----------
problems = []    # each problem: (file, spreadsheet row, what is wrong)  [row 1 is the header]
for i, r in meta.iterrows():
    row = i + 2
    for col in meta.columns:
        if pd.isna(r[col]) or r[col] == "":
            problems.append((r["file"], row, f"empty '{col}'"))
    if isinstance(r["condition"], str) and r["condition"] not in CONDITIONS and r["condition"] != "":
        problems.append((r["file"], row, f"unexpected condition label '{r['condition']}'"))
    ref = r["reference"]
    if isinstance(ref, str) and (ref.count("#") % 2 or ref.count("<") != ref.count(">")
                                 or ref.count("[") != ref.count("]") or ref.count("(") != ref.count(")")):
        problems.append((r["file"], row, "unbalanced annotation marks in reference (# < > [ ] ( ))"))

for i in meta.index[meta["file"].duplicated(keep="first") & meta["file"].notna()]:
    problems.append((meta.loc[i, "file"], i + 2, "duplicate file ID (this later copy is not used)"))
meta = meta.drop_duplicates(subset="file", keep="first").reset_index(drop=True)   # keep the first copy only

# ---------- open every audio file ----------
def find_audio(file_id):
    """Find the audio file; if the ID has no ending like .wav, try the usual ones."""
    if not isinstance(file_id, str):
        return None
    for ending in ["", ".wav", ".flac", ".mp3", ".m4a", ".ogg"]:
        path = os.path.join(AUDIO_DIR, file_id + ending)
        if os.path.isfile(path):
            return path
    return None

audio = {}      # file ID -> the audio (a long list of numbers)
for i, r in meta.iterrows():
    path = find_audio(r["file"])
    if path is None:
        problems.append((r["file"], i + 2, "audio file not found"))
        continue
    try:
        audio[r["file"]] = librosa.load(path, sr=16000, mono=True)[0]
    except Exception as e:
        problems.append((r["file"], i + 2, f"audio could not be opened ({type(e).__name__})"))

problems_df = pd.DataFrame(problems, columns=["file", "metadata_row", "problem"])
problems_df.to_csv(f"{OUTPUT_DIR}/data_check_problems.csv", index=False)

# ---------- print the summary ----------
print("Utterances in the metadata (after removing duplicate IDs):", len(meta))
print("Utterances whose audio loaded                           :", len(audio))
print("Number of speakers                                      :", meta["speaker"].nunique())
print("\nUtterances per condition:\n", meta["condition"].value_counts(dropna=False).to_string())
print("\nUtterances per speaker:\n", meta["speaker"].value_counts(dropna=False).sort_index().to_string())

# how many marks of each kind are in the references (a description only)
counts = []
for label, pattern in ANNOTATIONS.items():
    per_ref = [len(pattern.findall(t)) for t in meta["reference"].dropna()]
    counts.append({"Annotation": label, "Marks": sum(per_ref), "Utterances containing": sum(n > 0 for n in per_ref)})
print("\nAnnotation marks in the references:")
display(pd.DataFrame(counts))

print("MISSING OR INVALID ITEMS:")
display(problems_df) if len(problems_df) else print("None found.")

## 3. Loading the two ASR systems
I download System A and System B from Hugging Face, each with its processor (the part that prepares the audio for the model and turns the output back into text). Both are loaded in exactly the same way. If System B's page has no processor of its own, the standard Whisper-small processor is used instead and the code says so. I print which system is which.

In [ ]:
systems = {}
for key, model_id in SYSTEMS.items():
    try:
        processor = WhisperProcessor.from_pretrained(model_id)
    except Exception:
        print(f"No processor found for {model_id}; using the one from {SYSTEMS['A']} instead.")
        processor = WhisperProcessor.from_pretrained(SYSTEMS["A"])
    model = WhisperForConditionalGeneration.from_pretrained(model_id).to(DEVICE).eval()
    model.generation_config.forced_decoder_ids = None   # so BOTH models follow my GEN_SETTINGS only
    systems[key] = {"processor": processor, "model": model}

print("System A:", SYSTEMS["A"])
print("System B:", SYSTEMS["B"])
print("Transcription settings (both systems):", GEN_SETTINGS)

## 4. Transcribing every audio file with both systems
Each audio file is transcribed by System A and then by System B. Both receive exactly the same audio, and the reference transcripts are never given to either model. Nothing is typed in by hand. If a file fails, the error is recorded and shown (and saved in `transcription_failures.csv`) rather than skipped silently. Afterwards I put the original information and both outputs into one table.

In [ ]:
def transcribe(key, wave):
    """Give one recording to one system and return the text it writes."""
    system = systems[key]
    features = system["processor"](wave, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    with torch.no_grad():                                   # I am only using the model, not training it
        ids = system["model"].generate(features, **GEN_SETTINGS)
    return system["processor"].batch_decode(ids, skip_special_tokens=True)[0].strip()

hyp = {"A": {}, "B": {}}       # hypothesis = what each system wrote
failures = []
for n, (file_id, wave) in enumerate(audio.items(), start=1):
    for key in ["A", "B"]:     # the SAME audio goes to both systems
        try:
            hyp[key][file_id] = transcribe(key, wave)
        except Exception as e:
            failures.append((file_id, f"System {key}", f"{type(e).__name__}: {e}"))
    if n % 25 == 0 or n == len(audio):
        print(f"Transcribed {n} of {len(audio)} files")

failures_df = pd.DataFrame(failures, columns=["file", "system", "error"])
failures_df.to_csv(f"{OUTPUT_DIR}/transcription_failures.csv", index=False)
print("Transcription failures:", "none" if failures_df.empty else "")
if len(failures_df):
    display(failures_df)

# one row per utterance: original information + both outputs (empty if a file failed)
results = meta[["file", "speaker", "condition", "reference"]].copy()
results["system_a_hypothesis"] = results["file"].map(hyp["A"])
results["system_b_hypothesis"] = results["file"].map(hyp["B"])
results.head()

## 5. Cleaning (normalising) the text before scoring
Two transcripts can mean the same thing but look different to a computer ("Don't" vs "don't", "3" vs "three"), and I do not want such harmless differences counted as errors. I clean the text in two stages. **Stage 1 (references only):** `remove_marks()` removes my annotation marks (`#...#`, `<...>`, `[...]`, `(...)`) and keeps the words inside them, because the systems' outputs do not contain these marks. The result goes in a new column `reference_clean`; the original `reference` is never changed. **Stage 2 (reference and both outputs, identically):** `normalise_text()` (1) makes everything lowercase, (2) spells out contractions (can't → can not), (3) turns % and & into words, (4) turns digits into words (3 → three), (5) removes punctuation and hyphens, and (6) tidies spaces. I make no other changes, and I do not touch how particles are spelled. The original outputs are kept next to the cleaned versions because I need them for my later linguistic analysis.

In [ ]:
def remove_marks(text):
    """Stage 1 (references only): remove my annotation marks but keep the words inside them."""
    if pd.isna(text):
        return ""
    t = str(text)
    for pattern in ANNOTATIONS.values():
        t = pattern.sub(r" \1 ", t)            # keep what is inside, drop the marks (spaces stop words joining)
    t = re.sub(r"[#<>\[\]()]", " ", t)          # any single mark left over (e.g. an unmatched bracket)
    return re.sub(r"\s+", " ", t).strip()

# Contraction rules: (pattern, replacement), applied in this order (special cases first)
CONTRACTIONS = [
    (r"\bwon't\b", "will not"), (r"\bcan't\b", "can not"), (r"\bcannot\b", "can not"),
    (r"\bshan't\b", "shall not"), (r"\bain't\b", "is not"), (r"\blet's\b", "let us"),
    (r"\b(it|that|he|she|what|there|here|who|where|how|when)'s\b", r"\1 is"),   # "'s" only after these words
    (r"\b(\w+)n't\b", r"\1 not"), (r"\b(\w+)'re\b", r"\1 are"), (r"\b(\w+)'ve\b", r"\1 have"),
    (r"\b(\w+)'ll\b", r"\1 will"), (r"\b(\w+)'m\b", r"\1 am"), (r"\b(\w+)'d\b", r"\1 would"),
]

def number_to_words(match):
    """Turn a matched number ('3', '3.5', '2nd') into words."""
    s = match.group(0)
    try:
        ordinal = re.fullmatch(r"(\d+)(st|nd|rd|th)", s)
        if ordinal:
            return " " + num2words(int(ordinal.group(1)), to="ordinal") + " "
        return " " + num2words(float(s) if "." in s else int(s)) + " "
    except Exception:
        return s

def normalise_text(text):
    """Stage 2: the SAME cleaning for the reference and for both systems' outputs."""
    if pd.isna(text):
        return ""
    t = str(text).replace("\u2019", "'").replace("\u2018", "'").lower()       # curly -> straight apostrophes, lowercase
    for pattern, replacement in CONTRACTIONS:
        t = re.sub(pattern, replacement, t)                                    # spell out contractions
    t = t.replace("%", " percent").replace("&", " and ")                       # symbols -> words
    t = re.sub(r"(?<=\d),(?=\d{3})", "", t)                                    # 1,000 -> 1000
    t = re.sub(r"\d+(?:st|nd|rd|th)\b|\d+(?:\.\d+)?", number_to_words, t)      # digits -> words
    t = t.replace("-", " ")                                                    # twenty-one -> twenty one
    t = re.sub(r"[^\w\s]", "", t).replace("_", " ")                            # remove punctuation
    return re.sub(r"\s+", " ", t).strip()                                      # tidy spaces

# A quick look at what the two stages do (made-up examples, NOT from my test set)
for example in ["#Mary# say she go <kena> already [lah]", "I (um) can't go, it's 3pm [leh]!"]:
    cleaned = remove_marks(example)
    print(f"{example!r}\n   marks removed -> {cleaned!r}\n   then cleaned  -> {normalise_text(cleaned)!r}")

# ---------- apply to my data (original columns stay untouched) ----------
results["reference_clean"] = results["reference"].map(remove_marks)
results["reference_norm"]  = results["reference_clean"].map(normalise_text)
results["system_a_norm"]   = results["system_a_hypothesis"].map(normalise_text)
results["system_b_norm"]   = results["system_b_hypothesis"].map(normalise_text)

# Utterances that can be scored: both systems produced output and the cleaned reference has words
scored = results["system_a_hypothesis"].notna() & results["system_b_hypothesis"].notna() & (results["reference_norm"] != "")
eval_df = results[scored].reset_index(drop=True)
print(f"\nUtterances scored: {len(eval_df)} of {len(results)}")
print("Not scored (failed transcription, missing audio or empty reference):", len(results) - len(eval_df))

## 6. Word Error Rate (WER), overall, by condition and by speaker
WER measures how many word mistakes a system makes compared with the human reference: **WER = (S + D + I) / N**, where **S** = substituted words (a different word was written), **D** = deleted words (a word was left out), **I** = inserted words (an extra word was added) and **N** = number of words in the reference. `jiwer` lines up the reference and the output word by word and counts S, D, I and N; my code then calculates the WER from those counts and checks it against jiwer's own value. For each group (everything, each condition, each speaker) I add up the errors and reference words of all its utterances and then divide, so that longer utterances count for more. I report the number of utterances and reference words with every WER. This is only descriptive: I do not run any statistical tests.

In [ ]:
HYP = {"A": "system_a_norm", "B": "system_b_norm"}      # the column holding each system's cleaned output

def wer_stats(refs, hyps):
    """Count S, D, I, N for a group of utterances and calculate WER = (S + D + I) / N."""
    refs, hyps = list(refs), list(hyps)
    if not refs:
        return dict(utterances=0, ref_words=0, S=0, D=0, I=0, WER=float("nan"))
    out = jiwer.process_words(refs, hyps)               # jiwer lines up the words and counts errors
    S, D, I = out.substitutions, out.deletions, out.insertions
    N = out.substitutions + out.deletions + out.hits    # all words in the reference (hits = words correct)
    wer = (S + D + I) / N                               # <-- the WER formula
    assert abs(wer - out.wer) < 1e-9, "My WER does not match jiwer's"
    return dict(utterances=len(refs), ref_words=N, S=S, D=D, I=I, WER=wer)

# ---- overall ----
overall = pd.DataFrame([{"System": LABEL[k], **wer_stats(eval_df["reference_norm"], eval_df[HYP[k]])} for k in "AB"])

# ---- by condition ----
rows = []
for k in "AB":
    for cond in CONDITIONS:
        sub = eval_df[eval_df["condition"] == cond]
        rows.append({"System": LABEL[k], "Condition": cond, **wer_stats(sub["reference_norm"], sub[HYP[k]])})
by_condition = pd.DataFrame(rows)

# ---- by speaker ----
rows = []
for speaker, sub in eval_df.groupby("speaker"):
    a = wer_stats(sub["reference_norm"], sub[HYP["A"]])
    b = wer_stats(sub["reference_norm"], sub[HYP["B"]])
    rows.append({"Speaker": speaker, "Utterances": a["utterances"], "Reference words": a["ref_words"],
                 "System A WER": a["WER"], "System B WER": b["WER"],
                 "A_S": a["S"], "A_D": a["D"], "A_I": a["I"], "B_S": b["S"], "B_D": b["D"], "B_I": b["I"]})
by_speaker = pd.DataFrame(rows)

print("WER = (S + D + I) / N, calculated on the cleaned text\n")
print("Overall (with S, D, I, N):");  display(overall)
print("By condition:");               display(by_condition)
print("By speaker:");                 display(by_speaker)

## 7. Discourse-particle analysis (word-matching only)
For every target particle (*lah, leh, lor, meh, ah*) in a cleaned reference, I line up the reference with the system's output (the same lining-up jiwer uses for WER) and sort the particle into one of three groups: **recognised** (the system wrote exactly the same word in that position), **deleted** (nothing was written for it) or **substituted** (a different word was written; I save it for my later analysis). The **particle recognition rate** is recognised particles divided by the total target particles in the references, shown overall and for each particle. This is a strict spelling match only; I do not judge here whether another spelling or pronunciation counts as the same particle.

In [ ]:
def particle_tokens(key):
    """One row for every target particle in the references, saying what system `key` did with it."""
    col, rows = HYP[key], []
    for _, r in eval_df.iterrows():
        ref_words, hyp_words = r["reference_norm"].split(), r[col].split()
        targets = [i for i, w in enumerate(ref_words) if w in PARTICLES]
        if not targets:
            continue
        status, written = {}, {}
        if not hyp_words:                                   # the system wrote nothing at all
            for i in targets:
                status[i], written[i] = "deleted", ""
        else:
            # jiwer's word-by-word lining-up: chunks of type equal / delete / substitute / insert
            for ch in jiwer.process_words(r["reference_norm"], r[col]).alignments[0]:
                for i in range(ch.ref_start_idx, ch.ref_end_idx):
                    shown = " ".join(hyp_words[ch.hyp_start_idx:ch.hyp_end_idx])
                    if ch.type == "equal":
                        status[i], written[i] = "recognised", shown
                    elif ch.type == "delete":
                        status[i], written[i] = "deleted", ""
                    elif ch.type == "substitute":
                        status[i], written[i] = "substituted", shown
        for i in targets:
            rows.append({"system": LABEL[key], "file": r["file"], "speaker": r["speaker"],
                         "condition": r["condition"], "particle": ref_words[i], "ref_position": i + 1,
                         "status": status[i], "system_wrote": written[i]})
    return pd.DataFrame(rows, columns=["system", "file", "speaker", "condition", "particle",
                                       "ref_position", "status", "system_wrote"])

tokens = pd.concat([particle_tokens("A"), particle_tokens("B")], ignore_index=True)

# summary: recognised / deleted / substituted and the recognition rate, for each particle and for all together
rows = []
for k in "AB":
    for p in PARTICLES + ["ALL"]:
        sub = tokens[tokens["system"] == LABEL[k]]
        if p != "ALL":
            sub = sub[sub["particle"] == p]
        total, recognised = len(sub), int((sub["status"] == "recognised").sum())
        rows.append({"System": LABEL[k], "Particle": p, "Total in references": total, "Recognised": recognised,
                     "Deleted": int((sub["status"] == "deleted").sum()),
                     "Substituted": int((sub["status"] == "substituted").sum()),
                     "Recognition rate (%)": round(100 * recognised / total, 2) if total else float("nan")})
particle_summary = pd.DataFrame(rows)
display(particle_summary)

print("What each system wrote when a target particle was substituted (counts):")
subs = tokens[tokens["status"] == "substituted"]
display(subs.groupby(["system", "particle", "system_wrote"]).size().rename("count").reset_index()
        .sort_values(["system", "particle", "count"], ascending=[True, True, False]) if len(subs) else "None")

## 8. Tables and charts for the report
I arrange the numbers into Table 1 (overall WER), Table 2 (WER by condition) and Table 3 (WER by speaker), with WER as a percentage to two decimal places, and draw three bar charts: overall WER, WER by condition, and particle recognition rate per particle (with the number of occurrences, n, on each bar). The charts are saved as PNG files with the licence notice in small print underneath.

In [ ]:
# ---------- tables ----------
table1 = overall[["System", "utterances", "ref_words"]].rename(columns={"utterances": "Utterances", "ref_words": "Reference words"})
table1["WER (%)"] = (overall["WER"] * 100).round(2)

table2 = by_condition[["System", "Condition", "utterances", "ref_words"]].rename(columns={"utterances": "Utterances", "ref_words": "Reference words"})
table2["WER (%)"] = (by_condition["WER"] * 100).round(2)

table3 = by_speaker[["Speaker", "Utterances", "Reference words"]].copy()
table3["System A WER (%)"] = (by_speaker["System A WER"] * 100).round(2)
table3["System B WER (%)"] = (by_speaker["System B WER"] * 100).round(2)

print("Table 1: Overall WER");        display(table1)
print("Table 2: WER by condition");   display(table2)
print("Table 3: WER by speaker");     display(table3)

# ---------- charts ----------
colour = {"A": "#4C72B0", "B": "#DD8452"}                 # blue = System A, orange = System B
FOOTNOTE = LICENCE_NOTICE.replace(" from ", "\nfrom ").replace(" which is", "\nwhich is")   # split into lines to fit

def save_chart(fig, name):
    fig.text(0.5, -0.02, FOOTNOTE, ha="center", va="top", fontsize=5.5, color="gray")   # licence notice
    fig.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/{name}", dpi=200, bbox_inches="tight")
    plt.show()

def grouped_bars(ax, groups, values_by_system, labels, extra=None):
    """Draw blue/orange bars for System A and B for each group, with the value written on each bar."""
    width = 0.38
    for j, k in enumerate("AB"):
        xs = [i + (j - 0.5) * width for i in range(len(groups))]
        bars = ax.bar(xs, values_by_system[k], width, label=LABEL[k], color=colour[k])
        for n, (bar, v) in enumerate(zip(bars, values_by_system[k])):
            text = f"{v:.2f}" if extra is None else f"{v:.0f}\n(n={extra[n]})"
            ax.text(bar.get_x() + bar.get_width() / 2, v, text, ha="center", va="bottom", fontsize=8 if extra is None else 7)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(labels)

# Chart 1: overall WER
fig, ax = plt.subplots(figsize=(5, 4))
values = (overall["WER"] * 100).fillna(0).tolist()
bars = ax.bar(["System A\n(whisper-small)", "System B\n(sing2eng)"], values, color=[colour["A"], colour["B"]])
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom")
ax.set_ylabel("WER (%)"); ax.set_title("Overall WER"); ax.set_ylim(0, max(values + [1]) * 1.2)
save_chart(fig, "figure1_overall_wer.png")

# Chart 2: WER by condition
fig, ax = plt.subplots(figsize=(7, 4.5))
vals = {k: [by_condition[(by_condition["System"] == LABEL[k]) & (by_condition["Condition"] == c)]["WER"].iloc[0] * 100
            for c in CONDITIONS] for k in "AB"}
vals = {k: [0 if v != v else v for v in vals[k]] for k in "AB"}       # an empty group (nan) is drawn as 0
grouped_bars(ax, CONDITIONS, vals, CONDITIONS)
ax.set_ylabel("WER (%)"); ax.set_title("WER by condition"); ax.legend(fontsize=8)
ax.set_ylim(0, max(vals["A"] + vals["B"] + [1]) * 1.25)
save_chart(fig, "figure2_wer_by_condition.png")

# Chart 3: particle recognition rate
fig, ax = plt.subplots(figsize=(7, 4.5))
vals, ns = {}, []
for k in "AB":
    sub = particle_summary[(particle_summary["System"] == LABEL[k]) & (particle_summary["Particle"] != "ALL")].set_index("Particle").reindex(PARTICLES)
    vals[k] = sub["Recognition rate (%)"].fillna(0).tolist()
    ns = sub["Total in references"].astype(int).tolist()
grouped_bars(ax, PARTICLES, vals, PARTICLES, extra=ns)
ax.set_ylabel("Recognition rate (%)"); ax.set_title("Particle recognition rate (exact word match)")
ax.set_ylim(0, 125); ax.legend(fontsize=8)
save_chart(fig, "figure3_particle_recognition.png")

## 9. Saving the results
I save `results.csv` with exactly one row per test utterance and the columns `file, speaker, condition, reference, system_a_hypothesis, system_b_hypothesis` (original text, not the cleaned versions); the code checks that the number of rows matches my metadata. I also save a version with the cleaned text, the three tables, the detailed WER tables (with S, D, I), the particle results, and a text file with the licence notice. Everything is zipped and downloaded.

In [ ]:
# ---------- results.csv: one row per utterance, six columns ----------
results_out = results[["file", "speaker", "condition", "reference", "system_a_hypothesis", "system_b_hypothesis"]].fillna("")
assert len(results_out) == len(meta), "results.csv must have one row per utterance"
results_out.to_csv(f"{OUTPUT_DIR}/results.csv", index=False, encoding="utf-8-sig")

# ---------- extra files ----------
results.to_csv(f"{OUTPUT_DIR}/results_with_normalised_text.csv", index=False, encoding="utf-8-sig")
table1.to_csv(f"{OUTPUT_DIR}/table1_overall_wer.csv", index=False)
table2.to_csv(f"{OUTPUT_DIR}/table2_wer_by_condition.csv", index=False)
table3.to_csv(f"{OUTPUT_DIR}/table3_wer_by_speaker.csv", index=False)
overall.to_csv(f"{OUTPUT_DIR}/wer_detail_overall.csv", index=False)           # with S, D, I
by_condition.to_csv(f"{OUTPUT_DIR}/wer_detail_by_condition.csv", index=False)
by_speaker.to_csv(f"{OUTPUT_DIR}/wer_detail_by_speaker.csv", index=False)
particle_summary.to_csv(f"{OUTPUT_DIR}/particle_recognition_summary.csv", index=False)
tokens.to_csv(f"{OUTPUT_DIR}/particle_tokens.csv", index=False, encoding="utf-8-sig")
with open(f"{OUTPUT_DIR}/DATA_LICENCE_NOTICE.txt", "w", encoding="utf-8") as f:
    f.write(LICENCE_NOTICE + "\n")

# ---------- zip everything and download ----------
with zipfile.ZipFile("/content/asr_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(OUTPUT_DIR)):
        z.write(f"{OUTPUT_DIR}/{name}", arcname=name)
print("Saved:", sorted(os.listdir(OUTPUT_DIR)))

from google.colab import files
files.download("/content/asr_results.zip")

## Appendix: Singapore Open Data Licence Version 1.0 (full text)

*Latest version: https://data.gov.sg/open-data-licence*

**SINGAPORE OPEN DATA LICENCE VERSION 1.0**

**Acceptance & Grant of Licence**

By Using (as defined below) the datasets on the Relevant Websites, you agree to the terms of this Licence.

The Agency grants you a worldwide, perpetual, royalty-free, non-exclusive licence to Use the datasets, subject to the terms of this Licence.

**What You Can Do:**

You can use, access, download, copy, distribute, transmit, modify and adapt the datasets, or any derived analyses or applications, whether commercially or non-commercially (“Use”).

You may grant a sub-licence of the datasets if this is necessary to enable users of your application and/or website (“your Sub-Licensees”), to use your application or access your website.

If your Sub-Licensees require additional rights, your application and/or website should direct your Sub-Licensees to obtain the appropriate licence at Data.gov.sg.

**Limitations**

This licence does not grant you any rights over:
- any personal data in the dataset;
- third party rights that the Agency is not authorised to license; and
- patents, trademarks and design rights.

You must not Use the datasets in a way that suggests any official status or that an Agency endorses you or your Use of the datasets.

This Licence does not grant any rights to Downstream Sub-Licensees.

**Additional Conditions**

*Attribution:*

You must include in your products, applications or websites that Use the datasets, a conspicuous notice acknowledging the source of the datasets and including a link to the most recent version of this Licence. An example notice is provided below:

[Contains information from {name of dataset} accessed on {date of access of dataset} from {source of data} which is made available under the terms of the Singapore Open Data Licence version 1.0 {URL link to licence}]*

*Disclaimers:*

The datasets are provided on an “as is” and “as available” basis. The Agency makes no representations or warranties in relation to the datasets, including but not limited to any representation or warranty as to the accuracy, completeness, reliability, continued availability, timeliness, non-infringement, title, quality or fitness for any particular purpose of the datasets to the fullest extent permitted by the law.

To the extent permitted by law, the Agency shall not be liable to you or any third party whether in contract, tort (including negligence), restitution, breach of statutory duty or otherwise, for damage or loss of any kind arising directly or indirectly from your or any third party’s Use of, or inability to Use, the datasets or the Relevant Websites.

*Indemnity:*

You shall indemnify the Agency against all liabilities, damages, costs (including legal costs) and expenses arising directly or indirectly from:
- any claim made by a third party in connection with your Use of the datasets;
- your violation of any rights of another person; or
- any claim made by a third party in connection with the third party’s Use of the datasets or any derived analyses or applications which you have provided.

*Intellectual Property:*

All datasets are the intellectual property of the Agency.

You cannot enforce any intellectual property rights belonging to the Agency except with the prior written approval of the Agency. Rights of Third Parties: No one other than a party to this Licence shall have any right to enforce any of its terms.

*Termination of Licence:*

The Agency may terminate this Licence immediately upon your breach of any of the terms of this Licence. The Disclaimers and Indemnity sections, and any other provision of this Licence which is required to give effect to termination or the consequences of such termination, shall survive the termination of this Licence.

*Governing Law and Jurisdiction:*

This Licence is governed by the laws of the Republic of Singapore and the parties to the Licence submit to the exclusive jurisdiction of the Singapore courts.

**Definitions and Intepretation**

In this Licence:

“Agency” means the Singapore Government (including its Ministries, departments, and Organs of State) or the Statutory Board providing the dataset, and where the dataset derives its data from multiple Agencies, a reference to “Agency” shall include all the Agencies providing the source data.

“Dataset” includes any data, or information within the dataset, that is offered for use under the terms of this Licence.

“Downstream Sub-Licensee” means any person who may require a licence in order to access the website or use the application of a Sub-Licensee.

“Statutory Board” means a body incorporated by or under written law from time to time to perform or discharge any public function.

“Sub-Licensee” means a person that is granted a sub-licence by you, where allowed under this Licence.

“Relevant Websites” means this website or other websites owned by an Agency that are linked from this website.

**About the Singapore Open Data License**

The Singapore Open Data Licence aims to promote and enable easy reuse of Public Sector data to create value for the community and businesses.

This is version 1.0 of the Singapore Open Data Licence. The Singapore Government may, from time to time, issue new versions of the Singapore Open Data Licence. Your continued Use of the datasets constitutes your acceptance of the new version of this Licence.

Link to Singapore Open Data License version 1.0: https://data.gov.sg/open-data-licence